<center><h1>Keehan_Alyssa_HW3</h1></center>
<br>
<br>

**Name:** Alyssa Keehan 

**GitHub Username:** akeehan56

**GitHub Links:** [Class Repo](https://github.com/akeehan56/DSCI552) | [HW2](https://github.com/akeehan56/DSCI552/tree/main/HW2)

**Email:** akeehan@usc.edu  

**USC ID:** 3429542433

## 1. Time Series Classification Part 1: Feature Creation/Extraction

### (a) Download Data

Package imports

In [1]:
from pathlib import Path
import re

import pandas as pd

Get the AReM Data Set

In [12]:
# Change this path if your AReM folder is somewhere else
data_dir = Path("../data/AReM")

activities = [
    "bending1",
    "bending2",
    "cycling",
    "lying",
    "sitting",
    "standing",
    "walking"
]

columns = [
    "time",
    "avg_rss12",
    "var_rss12",
    "avg_rss13",
    "var_rss13",
    "avg_rss23",
    "var_rss23"
]

### (b) Test and Train Data

In [13]:
# Required test datasets for each activity
test_dataset_numbers = {
    "bending1": [1, 2],
    "bending2": [1, 2],
    "cycling": [1, 2, 3],
    "lying": [1, 2, 3],
    "sitting": [1, 2, 3],
    "standing": [1, 2, 3],
    "walking": [1, 2, 3]
}

In [10]:
train_instances = []
test_instances = []

for activity in activities:
    folder_path = data_dir / activity

    # Find and numerically sort all CSV files in the activity folder
    files = sorted(
        folder_path.glob("dataset*.csv"),
        key=lambda file: int(file.stem.replace("dataset", ""))
    )

    for file_path in files:
        dataset_num = int(file_path.stem.replace("dataset", ""))

        # Handles both comma-separated and space-separated files.
        # Only the first seven columns are retained because some files
        # contain an extra trailing delimiter.
        df = pd.read_csv(
            file_path,
            skiprows=5,
            header=None,
            names=columns,
            sep=r"[,\s]+",
            engine="python",
            usecols=range(7)
        )

        # Ensure that all values are numeric
        df = df.apply(pd.to_numeric, errors="raise")

        instance = {
            "activity": activity,
            "dataset": dataset_num,
            "file": file_path.name,
            "data": df
        }

        if dataset_num in test_dataset_numbers[activity]:
            test_instances.append(instance)
        else:
            train_instances.append(instance)

In [11]:
print("Training instances:", len(train_instances))
print("Testing instances:", len(test_instances))
print("Total instances:", len(train_instances) + len(test_instances))

Training instances: 69
Testing instances: 19
Total instances: 88


**Answer:** Datasets 1 and 2 from bending1 and bending2 were assigned to the test set. Datasets 1, 2, and 3 from each of the other activity folders were also assigned to the test set. All remaining datasets were assigned to the training set. This produced 69 training instances and 19 test instances.

### (c) Feature Extraction

#### i. Research

Common time-domain features used for time-series classification include:

- Measures of location: mean, median, minimum, maximum, and quantiles.
- Measures of variability: variance, standard deviation, range, and 
  interquartile range.
- Measures of distribution shape: skewness and kurtosis.
- Signal magnitude features: root mean square, absolute energy, and 
  mean absolute value.
- Change-based features: mean absolute change, maximum change, and 
  the number of zero or mean crossings.
- Temporal-dependence features: autocorrelation and partial autocorrelation.
- Pattern-based features: number of peaks, peak locations, and the lengths 
  of consecutive values above or below the mean.
- Trend features: slope, intercept, and other linear-trend measurements.

These features convert each time series into a fixed set of numerical values 
that can be used by standard classification algorithms.

#### ii. Extraction

In [14]:
activity_order = {
    activity: index
    for index, activity in enumerate(activities)
}

all_instances = sorted(
    train_instances + test_instances,
    key=lambda instance: (
        activity_order[instance["activity"]],
        instance["dataset"]
    )
)

print("Number of instances:", len(all_instances))

Number of instances: 88


In [16]:
time_series_columns = [
    "avg_rss12",
    "var_rss12",
    "avg_rss13",
    "var_rss13",
    "avg_rss23",
    "var_rss23"
]

feature_rows = []
feature_columns = []

for instance_number, instance in enumerate(all_instances, start=1):
    row = {
        "Instance": instance_number,
        "Activity": instance["activity"],
        "Dataset": instance["dataset"]
    }

    if instance["dataset"] in test_dataset_numbers[instance["activity"]]:
        row["Split"] = "test"
    else:
        row["Split"] = "train"

    for series_number, column in enumerate(time_series_columns, start=1):
        values = instance["data"][column]

        row[f"min{series_number}"] = values.min()
        row[f"max{series_number}"] = values.max()
        row[f"mean{series_number}"] = values.mean()
        row[f"median{series_number}"] = values.median()
        row[f"std{series_number}"] = values.std(ddof=1)
        row[f"1stQuart{series_number}"] = values.quantile(0.25)
        row[f"3rdQuart{series_number}"] = values.quantile(0.75)

        if instance_number == 1:
            feature_columns.extend([
                f"min{series_number}",
                f"max{series_number}",
                f"mean{series_number}",
                f"median{series_number}",
                f"std{series_number}",
                f"1stQuart{series_number}",
                f"3rdQuart{series_number}"
            ])

    feature_rows.append(row)

feature_dataset = pd.DataFrame(feature_rows)

feature_dataset = feature_dataset[
    ["Instance", "Activity", "Dataset", "Split"] + feature_columns
]

feature_dataset

,Instance,Activity,Dataset,Split,min1,max1,mean1,median1,std1,1stQuart1,...,std5,1stQuart5,3rdQuart5,min6,max6,mean6,median6,std6,1stQuart6,3rdQuart6
0,1,bending1,1,test,37.25,45.00,40.624792,40.50,1.476967,39.25,...,2.188449,33.0000,36.00,0.00,1.92,0.570583,0.43,0.582915,0.0000,1.3000
1,2,bending1,2,test,38.00,45.67,42.812812,42.50,1.435550,42.00,...,1.995255,32.0000,34.50,0.00,3.11,0.571083,0.43,0.601010,0.0000,1.3000
2,3,bending1,3,train,35.00,47.40,43.954500,44.33,1.558835,43.00,...,1.999604,35.3625,36.50,0.00,1.79,0.493292,0.43,0.513506,0.0000,0.9400
3,4,bending1,4,train,33.00,47.75,42.179812,43.50,3.670666,39.15,...,3.849448,30.4575,36.33,0.00,2.18,0.613521,0.50,0.524317,0.0000,1.0000
4,5,bending1,5,train,33.00,45.75,41.678063,41.75,2.243490,41.33,...,2.411026,28.4575,31.25,0.00,1.79,0.383292,0.43,0.389164,0.0000,0.5000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
83,84,walking,11,train,19.50,45.33,33.586875,34.25,4.650935,30.25,...,3.283983,13.7300,18.25,0.00,8.32,3.259729,3.11,1.640243,2.0500,4.3225
84,85,walking,12,train,19.75,45.50,34.322750,35.25,4.752477,31.00,...,3.119856,13.5000,17.75,0.00,9.67,3.432563,3.20,1.732727,2.1575,4.5650
85,86,walking,13,train,19.50,46.00,34.546229,35.25,4.842294,31.25,...,2.823124,14.0000,17.75,0.00,10.00,3.338125,3.08,1.656742,2.1600,4.3350
86,87,walking,14,train,23.50,46.25,34.873229,35.25,4.531720,31.75,...,3.131076,13.7500,18.00,0.00,9.51,3.424646,3.27,1.690960,2.1700,4.5000


In [18]:
print("Dataset shape:", feature_dataset.shape)
print("Number of extracted features:", len(feature_columns))

print(
    "Missing feature values:",
    feature_dataset[feature_columns].isna().sum().sum()
)

Dataset shape: (88, 46)
Number of extracted features: 42
Missing feature values: 0


#### iii. Standard Deviation

#### iv. Select Features

## 2. ISLR 3.7.4

### (a) Linear Train

### (b) Linear Test

### (c) Not Linear Train

### (d) Not Linear Testing

## 3. ISLR 3.7.3 - Extra Practice 

## 3. ISLR 3.7.5 - Extra Practice 